# Summary Statistics for Unnormalized Characteristics

This notebook loads both US and Global **final unnormalized output** files and presents summary statistics including:
- Total observations
- Valid share (share of non-null, non-NaN, non-inf observations)
- Invalid share (share of null, NaN, or inf observations)
- Mean (computed on valid observations only)
- Standard Deviation (computed on valid observations only)
- 5th Percentile (computed on valid observations only)
- 95th Percentile (computed on valid observations only)

**Data Sources:**
- `final_output_unnormalized_us.parquet` - US merged characteristics (yearly + price-based)
- `final_output_unnormalized_global.parquet` - Global merged characteristics (yearly + price-based)


## 1. Setup and Imports

In [1]:
import polars as pl
from pathlib import Path
import sys

# Add parent directory to path
sys.path.insert(0, str(Path("..").resolve()))

# Import 62 characteristics from constants
from data_collection.constants import CHARACTERISTIC_COLUMNS

# Data paths - relative to validation_notebooks/ folder
DATA_DIR = Path("../data/outputs")

# Final unnormalized output files (merged yearly + price-based characteristics)
US_PATH = DATA_DIR / "final_output_unnormalized_us.parquet"
GLOBAL_PATH = DATA_DIR / "final_output_unnormalized_global.parquet"

print("=== Data File Paths ===")
print(f"US unnormalized: {US_PATH} (exists: {US_PATH.exists()})")
print(f"Global unnormalized: {GLOBAL_PATH} (exists: {GLOBAL_PATH.exists()})")

=== Data File Paths ===
US unnormalized: ..\data\outputs\final_output_unnormalized_us.parquet (exists: True)
Global unnormalized: ..\data\outputs\final_output_unnormalized_global.parquet (exists: True)


In [2]:
# Use the 62 Freyberger characteristics from constants
ALL_CHAR_NAMES = CHARACTERISTIC_COLUMNS

print(f"Total characteristics to analyze: {len(ALL_CHAR_NAMES)}")
print(f"\nFirst 10 characteristics: {ALL_CHAR_NAMES[:10]}")
print(f"Last 10 characteristics: {ALL_CHAR_NAMES[-10:]}")

Total characteristics to analyze: 62

First 10 characteristics: ['r2_1', 'r6_2', 'r12_2', 'r12_7', 'r36_13', 'Investment', 'ACEQ', 'DPI2A', 'AShrout', 'IVC']
Last 10 characteristics: ['LME', 'LME_adj', 'LTurnover', 'Rel2High', 'Ret_max', 'Spread', 'Std_Turn', 'Std_Vol', 'SUV', 'Total_vol']


In [3]:
def compute_summary_stats(df: pl.DataFrame, char_cols: list[str]) -> pl.DataFrame:
    """
    Compute summary statistics for characteristics.

    Filters out invalid observations (null, NaN, inf) before computing statistics.

    Args:
        df: DataFrame with characteristics
        char_cols: List of characteristic column names

    Returns:
        DataFrame with summary statistics including:
        - total_obs: Total number of observations
        - N: Number of valid observations (non-null, non-NaN, non-inf)
        - start_date: First date with a valid observation
        - end_date: Last date with a valid observation
        - valid_share: Share of valid observations
        - invalid_share: Share of invalid observations
        - mean, std, p5, p95: Computed on valid observations only
    """
    has_date = "date" in df.columns

    stats = []
    for col in char_cols:
        if col in df.columns:
            col_data = df[col]
            total_obs = col_data.len()

            # Mask for valid observations (non-null, non-NaN, non-inf)
            valid_mask = (
                col_data.is_not_null() & col_data.is_not_nan() & col_data.is_finite()
            )

            # Filter to valid observations only
            valid_data = col_data.filter(valid_mask)
            valid_obs = valid_data.len()
            invalid_obs = total_obs - valid_obs

            # Compute shares
            valid_share = valid_obs / total_obs if total_obs > 0 else 0.0
            invalid_share = invalid_obs / total_obs if total_obs > 0 else 0.0

            # Date range of valid observations
            if has_date and valid_obs > 0:
                valid_dates = df.filter(valid_mask).select("date")
                start_date = valid_dates.select(pl.col("date").min()).item()
                end_date = valid_dates.select(pl.col("date").max()).item()
            else:
                start_date = None
                end_date = None

            # Compute statistics on valid data only
            stats.append(
                {
                    "characteristic": col,
                    "total_obs": total_obs,
                    "N": valid_obs,
                    "start_date": start_date,
                    "end_date": end_date,
                    "valid_share": valid_share,
                    "invalid_share": invalid_share,
                    "mean": valid_data.mean() if valid_obs > 0 else None,
                    "std": valid_data.std() if valid_obs > 0 else None,
                    "p5": valid_data.quantile(0.05) if valid_obs > 0 else None,
                    "p95": valid_data.quantile(0.95) if valid_obs > 0 else None,
                }
            )
        else:
            stats.append(
                {
                    "characteristic": col,
                    "total_obs": 0,
                    "N": 0,
                    "start_date": None,
                    "end_date": None,
                    "valid_share": 0.0,
                    "invalid_share": 0.0,
                    "mean": None,
                    "std": None,
                    "p5": None,
                    "p95": None,
                }
            )
    return pl.DataFrame(stats).to_pandas().set_index("characteristic").sort_index()

## 2. Load Data

In [4]:
# Load US unnormalized final output
if US_PATH.exists():
    df_us = pl.read_parquet(US_PATH)
    print(f"US data loaded: {df_us.shape[0]:,} rows x {df_us.shape[1]} columns")
else:
    df_us = None
    print("US unnormalized data file not found!")

# Load Global unnormalized final output
if GLOBAL_PATH.exists():
    df_global = pl.read_parquet(GLOBAL_PATH)
    print(
        f"Global data loaded: {df_global.shape[0]:,} rows x {df_global.shape[1]} columns"
    )
else:
    df_global = None
    print("Global unnormalized data file not found!")

US data loaded: 975,210 rows x 65 columns
Global data loaded: 7,827,717 rows x 66 columns


## 3. US Summary Statistics (Unnormalized)

In [5]:
if df_us is not None:
    # Get characteristics that exist in the data
    us_available_chars = [c for c in ALL_CHAR_NAMES if c in df_us.columns]
    print(
        f"Available characteristics in US data: {len(us_available_chars)} / {len(ALL_CHAR_NAMES)}"
    )
    missing_chars = [c for c in ALL_CHAR_NAMES if c not in df_us.columns]
    if missing_chars:
        print(f"Missing characteristics ({len(missing_chars)}): {missing_chars}")

    us_stats = compute_summary_stats(df_us, us_available_chars)
    pl.Config.set_tbl_rows(65)
    us_stats.to_csv("summary_us.csv")
    display(us_stats)
else:
    print("Cannot compute US statistics - data not loaded")

Available characteristics in US data: 62 / 62


,total_obs,N,start_date,end_date,valid_share,invalid_share,mean,std,p5,p95
characteristic,,,,,,,,,,
A2ME,975210,921173,1988-07-29,2021-06-30,0.944589,0.055411,5.877977,447.476111,0.209230,10.681073
ACEQ,975210,815117,1988-07-29,2021-06-30,0.835837,0.164163,-7.398065,1738.234540,-0.863596,2.707702
AOA,975210,640900,1988-07-29,2021-06-30,0.657192,0.342808,211.840265,1194.391131,0.210000,790.000000
ASO,975210,805269,1988-07-29,2021-06-30,0.825739,0.174261,0.148043,0.611709,-0.069926,0.763081
AShrout,975210,805304,1988-07-29,2021-06-30,0.825775,0.174225,97.836865,7028.886416,-0.091679,1.561728
...,...,...,...,...,...,...,...,...,...,...
r12_2,975210,914203,1988-07-29,2021-06-30,0.937442,0.062558,0.148776,0.812598,-0.602205,1.126439
r12_7,975210,915897,1988-07-29,2021-06-30,0.939179,0.060821,0.068182,0.485161,-0.487937,0.721534
r2_1,975210,966979,1988-07-29,2021-06-30,0.991560,0.008440,0.014712,0.184291,-0.210526,0.256426


## 4. Global Summary Statistics (Unnormalized)

In [6]:
if df_global is not None:
    global_available_chars = [c for c in ALL_CHAR_NAMES if c in df_global.columns]
    print(
        f"Available characteristics in Global data: {len(global_available_chars)} / {len(ALL_CHAR_NAMES)}"
    )
    missing_chars = [c for c in ALL_CHAR_NAMES if c not in df_global.columns]
    if missing_chars:
        print(f"Missing characteristics ({len(missing_chars)}): {missing_chars}")

    global_stats = compute_summary_stats(df_global, global_available_chars)
    display(global_stats)
else:
    print("Cannot compute Global statistics - data not loaded")

Available characteristics in Global data: 62 / 62


,total_obs,N,start_date,end_date,valid_share,invalid_share,mean,std,p5,p95
characteristic,,,,,,,,,,
A2ME,7827717,7262583,1988-07-31,2022-06-30,0.927803,0.072197,0.001784,1.980525e-01,1.873091e-07,0.000047
ACEQ,7827717,7386036,1988-07-31,2022-06-30,0.943575,0.056425,2313.368384,1.064424e+06,-3.293460e-01,1.305935
AOA,7827717,5710744,1989-01-31,2022-06-30,0.729554,0.270446,111915.147317,4.900084e+06,9.120000e-01,28003.000000
ASO,7827717,6771675,1988-07-31,2022-06-30,0.865089,0.134911,0.073582,4.588906e-01,-7.750531e-03,0.337711
AShrout,7827717,6771689,1988-07-31,2022-06-30,0.865091,0.134909,27.925985,3.225341e+03,-8.863410e-03,0.856545
...,...,...,...,...,...,...,...,...,...,...
r12_2,7827717,7415510,1988-07-31,2022-06-30,0.947340,0.052660,20136.444021,1.046192e+07,-6.130952e-01,1.512118
r12_7,7827717,7488529,1988-07-31,2022-06-30,0.956668,0.043332,18484.215840,1.394149e+07,-4.755574e-01,0.892368
r2_1,7827717,7774973,1988-07-31,2022-06-30,0.993262,0.006738,0.144562,8.431183e+01,-1.773962e-01,0.217290


## 5. Comparison: US vs Global

In [7]:
if df_us is not None and df_global is not None:
    print("=== Comparison Summary ===")
    print(f"\nUS: {len(us_available_chars)}/62 characteristics available")
    print(f"Global: {len(global_available_chars)}/62 characteristics available")

    # Find characteristics present in both
    common_chars = set(us_available_chars).intersection(set(global_available_chars))
    print(f"\nCommon characteristics: {len(common_chars)}")

    # US-only and Global-only
    us_only = set(us_available_chars) - set(global_available_chars)
    global_only = set(global_available_chars) - set(us_available_chars)

    if us_only:
        print(f"US-only: {list(us_only)}")
    if global_only:
        print(f"Global-only: {list(global_only)}")
else:
    print("Cannot compare - one or both datasets not loaded")

=== Comparison Summary ===

US: 62/62 characteristics available
Global: 62/62 characteristics available

Common characteristics: 62


In [8]:
us_stats.sort_values(by="invalid_share")

,total_obs,N,start_date,end_date,valid_share,invalid_share,mean,std,p5,p95
characteristic,,,,,,,,,,
AT_raw,975210,975210,1988-07-29,2021-06-30,1.000000,0.000000,8.614327e+03,7.272614e+04,4.883000e+00,2.163263e+04
LME_adj,975210,975210,1988-07-29,2021-06-30,1.000000,0.000000,6.674284e-04,2.025457e+07,-7.970655e+06,6.772900e+06
LME,975210,975210,1988-07-29,2021-06-30,1.000000,0.000000,3.183117e+06,2.057862e+07,4.558120e+03,1.154998e+07
C,975210,971429,1988-07-29,2021-06-30,0.996123,0.003877,1.849728e-01,2.308540e-01,3.641609e-03,7.418408e-01
SAT_adj,975210,967508,1988-07-29,2021-06-30,0.992102,0.007898,-1.044917e-01,3.569155e+00,-1.070047e+00,1.020703e+00
...,...,...,...,...,...,...,...,...,...,...
Beta,975210,670902,1988-08-31,2021-06-30,0.687956,0.312044,7.837780e-01,6.297141e-01,-9.336457e-02,1.846743e+00
Beta_Cor,975210,670888,1988-08-31,2021-06-30,0.687942,0.312058,2.648234e-01,2.059197e-01,-2.183562e-02,6.245004e-01
Free_CF,975210,648810,1988-07-29,2021-06-30,0.665303,0.334697,-7.068083e-01,1.013302e+02,-1.284531e+00,4.684822e-01
